# Procesamiento de datos mediante Apache Spark

## Autores

Eduardo Alarcón Navarro - 100472175@alumnos.uc3m.es

Ángela Ochoa Lavieja - 100562058@alumnos.uc3m.es

Lucía Velasco González - 100383335@alumnos.uc3m.es

Manuel Gómez-Plana Rodríguez - 100472310@alumnos.uc3m.es

# 1. Introducción

Este documento guarda la información relacionada con la elaboración de la práctica 1 de la asignatura Computación de Altas Prestaciones del Máster Universitario en Ciencia y Tecnología Informática. El notebook se divide en las siguientes secciones:

1. **Introducción**: se introduce la práctica.
2. **Problemas**: se introducen los posibles problemas planteados para realizar el estudio y el problema escogido para su análisis.
3. **Resolución**: se resuelve el problema mediante tres técnicas distintas: consultas SQL, concatenación de dataframes de Pandas y uso de RDDs.
4. **Análisis**: se analizan los resultados obtenidos de los tres problemas para determinar aspectos como la solución más rápida, el speedup obtenido mediante cada técnica o la escalabilidad de las soluciones.
5. **Conclusiones**: se concluye el trabajo con unos pensamientos finales.

# Problemas



## Problemas Planteados

## Problema escogido

# Resolución

In [1]:
# Importado de Spark
from pyspark.sql import SparkSession
from pyspark import SparkContext, SparkConf
from pyspark.sql.functions import *
import pandas as pd
import numpy as np

import time

# Creación de la sesión de spark
spark = SparkSession.builder.master('local[*]').getOrCreate()
#spark.conf.set('spark.cores.max', '1')
#spark.conf.set('spark.driver.memory','1000M')

sc = spark.sparkContext
sc

<SparkContext master=local[*] appName=pyspark-shell>

In [2]:
zone_map = spark.read.format("csv").option("inferSchema", "true").option("header", "true").option("mode", "DROPMALFORMED").load("content/taxi_zone_lookup.csv")
zone_map.show()
# zone_map: dict[int, list[str]] = zone_map.toPandas().set_index('LocationID').T.to_dict('list')

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
|        11|     Brooklyn|          Bath Beach|   Boro Zone|
|        12|    Manhattan|        Battery Park| Yellow Zone|
|        13|    Manhattan|   Battery Park City| Yellow Zone|
|        14|     Brookly

In [3]:
df = spark.read.format("parquet").option("inferSchema", "true").option("timestampFormat","yyyy-MM-dd HH:mm:ss").option("header", "true").option("mode", "DROPMALFORMED").load("content/yellow_tripdata_2025-01.parquet")
df.printSchema()
df.show(5)
type(df)

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)

+--------+--------------------+---------------------+---------------+------

pyspark.sql.classic.dataframe.DataFrame

Limpieza de datos

In [4]:
# df = df.na.drop()

In [5]:
df = df.filter(df.trip_distance != 0)
df = df.filter(df.tpep_pickup_datetime != df.tpep_dropoff_datetime)

In [6]:
start_sql = time.time()
TABLE_NAME = 'taxi_routes'
SQL = f"""SELECT HOUR(tpep_pickup_datetime) AS hour,
            ROUND(AVG(trip_distance / DATE_DIFF(SECOND,tpep_pickup_datetime,tpep_dropoff_datetime))*1.6*3600, 2) as vel_media_km_h 
            FROM {TABLE_NAME} 
            GROUP BY hour 
            SORT BY hour """

SQL = f"""SELECT HOUR(tpep_pickup_datetime) AS hour,
            AVG(trip_distance / DATE_DIFF(SECOND,tpep_pickup_datetime,tpep_dropoff_datetime))*1.6*3600 as vel_media_km_h 
            FROM {TABLE_NAME} 
            GROUP BY hour 
            SORT BY hour """

df.createOrReplaceTempView(TABLE_NAME)
sql_result = spark.sql(SQL)
sql_result.show(24)
end_sql = time.time()
print(f'Time taken: {end_sql-start_sql:.2f} seconds')

+----+------------------+
|hour|    vel_media_km_h|
+----+------------------+
|   0| 38.99589768198895|
|   1|  39.9114420518163|
|   2|23.365632414324825|
|   3|24.476499472191215|
|   4| 45.45207353215525|
|   5| 658.9031654073427|
|   6|180.15640763308664|
|   7|43.734545039262215|
|   8| 68.31087718860877|
|   9| 45.10769366185081|
|  10| 29.38894794236935|
|  11| 16.25490023710596|
|  12| 17.43804025837124|
|  13|27.220104765837014|
|  14| 47.11974083699504|
|  15|28.209201955683746|
|  16| 32.58625704216827|
|  17| 36.57332957427586|
|  18|31.951676771058143|
|  19|28.104021698617807|
|  20|30.745506345126824|
|  21| 64.85060771642023|
|  22|33.279364938472476|
|  23|40.287091662018184|
+----+------------------+

Time taken: 0.84 seconds


## Usando dataframes

In [7]:
start_dataframe = time.time()
df1 = df.select('tpep_dropoff_datetime','tpep_pickup_datetime','trip_distance')
df2 = df1.withColumn('DiffInSeconds', (col('tpep_dropoff_datetime') - col('tpep_pickup_datetime')).cast('long'))
df3 = df2.withColumn('vel_km_h', try_divide('trip_distance','DiffInSeconds')*5760)
df4 = df3.withColumn('start_hour', hour(col('tpep_pickup_datetime')))
df5 = df4.select('start_hour', 'vel_km_h').groupBy("start_hour").avg('vel_km_h').sort('start_hour')
df5.show(truncate=False)
end_dataframe = time.time()
print(f'Time taken: {end_dataframe-start_dataframe:.2f} seconds')


+----------+------------------+
|start_hour|avg(vel_km_h)     |
+----------+------------------+
|0         |38.99589768198912 |
|1         |39.911442051816415|
|2         |23.365632414324814|
|3         |24.476499472191247|
|4         |45.452073532155325|
|5         |658.9031654073425 |
|6         |180.1564076330861 |
|7         |43.73454503926236 |
|8         |68.31087718860894 |
|9         |45.10769366185122 |
|10        |29.388947942369864|
|11        |16.25490023710637 |
|12        |17.438040258371604|
|13        |27.220104765837434|
|14        |47.119740836995994|
|15        |28.20920195568426 |
|16        |32.58625704216881 |
|17        |36.5733295742762  |
|18        |31.951676771058633|
|19        |28.10402169861853 |
+----------+------------------+
only showing top 20 rows
Time taken: 0.39 seconds


Usando RDD

In [8]:
# m1 = df.select('tpep_dropoff_datetime', 'tpep_pickup_datetime', 'trip_distance').rdd.map(lambda x: (hour(x[1]), 5760*x[2]/(x[0]-x[1])))m1 = m1.
start_rdd = time.time()
m1 = df.select('tpep_dropoff_datetime', 'tpep_pickup_datetime', 'trip_distance').rdd.map(lambda x: (x[1].hour, (x[2]/(x[0]-x[1]).seconds,1)))
r1 = m1.reduceByKey(lambda x,y: (x[0] + y[0], x[1] + y[1]))
m2 = r1.map(lambda x: (x[0], 5760*x[1][0]/x[1][1])).sortBy(lambda x: x[0])
end_tasks = time.time()
result = m2.take(24)
end_rdd = time.time()
print(f'Time taken: {end_tasks-start_rdd:.2f} seconds only creating tasks')
print(f'Time taken: {end_rdd-start_rdd:.2f} seconds with bringing data to python')


Time taken: 3.09 seconds only creating tasks
Time taken: 3.39 seconds with bringing data to python


In [9]:
m2.take(24)

[(0, 39.292093938822624),
 (1, 39.91145762328096),
 (2, 23.401732053083663),
 (3, 24.522011276223306),
 (4, 45.532578504253294),
 (5, 659.2789061634846),
 (6, 180.26993429014945),
 (7, 43.74871777034736),
 (8, 68.3727812352505),
 (9, 45.61844700783106),
 (10, 29.942264985332073),
 (11, 17.09648885684142),
 (12, 17.680425525828554),
 (13, 27.387515832640705),
 (14, 47.19171423588695),
 (15, 28.285050764472857),
 (16, 32.65225041407704),
 (17, 36.60746144973351),
 (18, 31.99644588142017),
 (19, 28.104021698617803),
 (20, 30.745506345126824),
 (21, 64.91120022794408),
 (22, 33.8534056686105),
 (23, 40.424224808062654)]

# Análisis de Prueba 2: Qué barrio / zona da más propinas
(en el lugar de recogida del cliente)


map: LocationID, Borough, Zone, service_zone

In [10]:
start_dataframe = time.time()

df_propina = df.select('PULocationID', 'tip_amount', 'tpep_pickup_datetime')
df_propina_1 = df_propina.withColumn('start_hour', hour(col('tpep_pickup_datetime')))
df_propina_2 = df_propina_1.groupBy('PULocationID','start_hour').sum('tip_amount')
zone_map_red_1 = zone_map.select('LocationID', 'Borough', 'Zone')
zone_map_red_2 = zone_map_red_1.withColumn('PULocationID', col('LocationID')).select('PULocationID','Borough','Zone')
df_propina_3 = df_propina_2.join(zone_map_red_2, on='PULocationID').withColumn('tip_sum', round('sum(tip_amount)', 3)).select('start_hour', 'Borough', 'Zone', 'tip_sum')
df_propina_4 = df_propina_3.sort('tip_sum', ascending=False)
df_propina_4.show(n=30)

end_dataframe = time.time()
print(f'Time taken: {end_dataframe-start_dataframe:.2f} seconds')

+----------+---------+-----------------+---------+
|start_hour|  Borough|             Zone|  tip_sum|
+----------+---------+-----------------+---------+
|        16|   Queens|      JFK Airport|114814.04|
|        15|   Queens|      JFK Airport| 97955.33|
|        19|   Queens|      JFK Airport| 95860.98|
|        20|   Queens|      JFK Airport| 92614.27|
|        21|   Queens|      JFK Airport| 91546.48|
|        17|   Queens|      JFK Airport| 91287.72|
|        22|   Queens|      JFK Airport| 83437.37|
|        18|   Queens|      JFK Airport| 78920.46|
|        14|   Queens|      JFK Airport| 74116.38|
|        23|   Queens|      JFK Airport| 66629.46|
|        14|   Queens|LaGuardia Airport| 64621.18|
|        15|   Queens|LaGuardia Airport| 58581.23|
|        16|   Queens|LaGuardia Airport| 57071.29|
|        13|   Queens|      JFK Airport|  53251.2|
|        13|   Queens|LaGuardia Airport| 51935.68|
|        17|   Queens|LaGuardia Airport| 51683.46|
|        21|   Queens|LaGuardia

# Análisis de Prueba 3: Tipo de pago por rango de precio
Cómo me van a pagar dependiendo de la cantidad de gente que haya y un rago de dinero.

In [31]:
from pyspark.ml.feature import QuantileDiscretizer
from pyspark.sql.types import *

NUMBER_BUCKETS = 43000
start_dataframe = time.time()
# payment_type = pt
df_pt1 = df.select('payment_type', 'total_amount', 'passenger_count')
df_pt2 = df_pt1.filter(df_pt1.total_amount > 0).filter(col('passenger_count').isNotNull())

qds1 = QuantileDiscretizer(inputCol="total_amount", outputCol="total_amount_bucket", numBuckets=NUMBER_BUCKETS)
df_pt2 = qds1.fit(df_pt2).transform(df_pt2)

# a = df.select(max(df.total_amount))

max_payment, min_payment = df_pt2.select(max(df_pt2.total_amount), min(df_pt2.total_amount)).toPandas().T.to_dict('list')[0]

factor_bucket = (max_payment - min_payment) / NUMBER_BUCKETS 

def aux(valor):
    #return (f"{round(valor*factor_bucket, 2)}-{round((valor+1)*factor_bucket, 2)}")
    min_val = np.round(valor*factor_bucket)
    max_val = np.round(min_val + factor_bucket)
    print(type(min_val), min_val)
    return (f"{min_val}-{max_val}")

udf_ = udf(aux, StringType())

df_pt3 = df_pt2.groupBy('total_amount_bucket').agg(mode('payment_type'), mode('passenger_count'))
df_pt4 = df_pt3.withColumn('price_ranges', udf_(df_pt3['total_amount_bucket'])).sort('total_amount_bucket')
df_pt5 = df_pt4.select('price_ranges', 'mode(passenger_count)', 'mode(payment_type)')

df_pt5.show(n=NUMBER_BUCKETS+1)

end_dataframe = time.time()
print(f'Time taken: {end_dataframe-start_dataframe:.2f} seconds')

+---------------+---------------------+------------------+
|   price_ranges|mode(passenger_count)|mode(payment_type)|
+---------------+---------------------+------------------+
|      20.0-40.0|                    1|                 2|
|      40.0-60.0|                    1|                 2|
|      60.0-80.0|                    1|                 2|
|     80.0-100.0|                    1|                 2|
|    100.0-120.0|                    1|                 2|
|    120.0-140.0|                    1|                 2|
|    141.0-161.0|                    1|                 2|
|    161.0-181.0|                    1|                 1|
|    181.0-201.0|                    1|                 2|
|    201.0-221.0|                    1|                 1|
|    221.0-241.0|                    1|                 2|
|    241.0-261.0|                    1|                 1|
|    261.0-281.0|                    1|                 1|
|    281.0-301.0|                    1|                 

# Análisis

# Conclusiones

## Esto te molesta? te importa?

14| 47.11974083699504

14| 47.119740836995994

14| 47.19171423588695

¿Hemos decidido que los viajes están indexados por la hora de comienzo del viaje?

¿Te molesta que hagamos los estudios que nos dijiste en clase?

# Resultados temporales

En orden, sql, dataframe, rdd:
Con 1 cores:
Time taken: 0.77 seconds
Time taken: 0.47 seconds
Time taken: 0.04 seconds only creating tasks
Time taken: 7.16 seconds with bringing data to python

Con 14 core.
Time taken: 0.30 seconds
Time taken: 0.25 seconds
Time taken: 2.64 seconds only creating tasks
Time taken: 2.88 seconds with bringing data to python